# Colombia Open — Infraestructura (puntos de interés)

Este notebook muestra cómo cargar el dataset **`infraestructura_poi`** (puntos de interés urbanos — salud, educación, seguridad, finanzas, religión, gobierno, transporte, comercio — geocodificados por municipio, OpenStreetMap vía Geofabrik) con `colombia-open`.

> **Qué es y qué no es:** son los POI que la comunidad de OpenStreetMap mapeó para Colombia, clasificados en una lista curada de categorías cívicas. Refleja qué tan mapeado está un lugar, no necesariamente qué infraestructura existe de verdad — ver Limitaciones.

> **Nota (2026-10):** `colombia-open` está publicado en [PyPI](https://pypi.org/project/colombia-open/) — `pip install colombia-open` descarga y cachea el bundle de datos publicado en GitHub Releases automáticamente. Este dataset está disponible desde el bundle `v0.9.0`.

## Setup

In [ ]:
!pip install -q colombia-open

## Cargar el dataset

In [ ]:
import polars as pl
from colombia_open import ColombiaOpen

hub = ColombiaOpen()
poi = hub.load_polars("infraestructura_poi")

print(poi.shape)
poi.head(10)

## Esquema

| Columna | Tipo | Descripción |
|---|---|---|
| `osm_id` | int | ID del nodo en OpenStreetMap — clave primaria, única. |
| `categoria` | string | `salud`, `educacion`, `seguridad`, `finanzas`, `religion`, `gobierno`, `transporte` o `comercio`. |
| `subcategoria` | string | Valor original de OpenStreetMap (`amenity`/`shop`), ej. "pharmacy", "school". |
| `nombre` | string, nullable | Etiqueta `name` del POI — ~9% de los nodos no la traen. |
| `cod_dpto` / `cod_mpio` | string (2 / 5 dígitos) | Derivados de un join espacial contra `municipios_geometria`. |
| `longitud` / `latitud` | float | EPSG:4326. |

Esquema completo: [`docs/datasets/infraestructura_poi.md`](../../docs/datasets/infraestructura_poi.md).

## POI por categoría, nacional

In [ ]:
poi.group_by("categoria").agg(pl.len().alias("n_poi")).sort("n_poi", descending=True)

## Acceso a salud por 100 mil habitantes (join con `demografia`)

In [ ]:
poblacion = hub.load_polars("demografia").filter(
    (pl.col("area_geografica") == "Total") & (pl.col("anio") == 2024)
)

(
    poi.filter(pl.col("categoria") == "salud")
    .group_by("cod_mpio")
    .agg(pl.len().alias("poi_salud"))
    .join(poblacion.select(["cod_mpio", "nombre_mpio", "poblacion"]), on="cod_mpio")
    .with_columns((pl.col("poi_salud") / pl.col("poblacion") * 100_000).round(2).alias("poi_salud_100k"))
    .filter(pl.col("poblacion") > 100_000)
    .sort("poi_salud_100k", descending=True)
    .head(10)
)

## Farmacias por municipio

In [ ]:
(
    poi.filter(pl.col("subcategoria") == "pharmacy")
    .group_by("cod_mpio")
    .agg(pl.len().alias("farmacias"))
    .sort("farmacias", descending=True)
    .head(10)
)

## Limitaciones conocidas

- **Refleja cobertura de mapeo, no infraestructura real.** Un municipio sin POI de una categoría puede no tenerla, o puede no estar mapeada todavía (OSM es de edición voluntaria, cobertura desigual urbano/rural).
- **Solo nodos (puntos), no polígonos de edificios** — un supermercado o colegio mapeado como el contorno de su edificio no se incluye.
- **Categorías curadas**, no todo `amenity`/`shop` de OpenStreetMap.
- **~450 POI excluidos por geocodificación** (zonas de frontera/costeras, o efecto de la simplificación de `municipios_geometria`).
- **Snapshot de un momento dado** — el extracto de Geofabrik se regenera diariamente, pero el dataset publicado queda fijo a la fecha del build.

Documentación completa: [`docs/datasets/infraestructura_poi.md`](../../docs/datasets/infraestructura_poi.md).